# Ford Car Price Prediction

This notebook builds and compares **two linear regression models** for predicting Ford used car prices:
- **Model 1**: One-Hot Encoding for categorical features
- **Model 2**: Label Encoding for categorical features

We compare them using R² and Adjusted R² to determine which encoding strategy works better.

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Dataset path (Kaggle environment)
df = pd.read_csv(r"/kaggle/input/datasets/adhurimquku/ford-car-price-prediction/ford.csv")
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# First look at the data
df.head()

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

### 2.1 Distribution of Target Variable (Price)

In [ ]:
sns.histplot(df["price"], bins=50, kde=True)
plt.title("Distribution of Car Prices")
plt.xlabel("Price")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

### 2.2 Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

### 2.3 Price by Year

In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(data=df, x='year', y='price')
plt.xticks(rotation=90)
plt.title("Price Distribution by Year")
plt.tight_layout()
plt.show()

### 2.4 Price vs Mileage

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='mileage', y='price', alpha=0.4)
plt.title("Price vs Mileage")
plt.tight_layout()
plt.show()

### 2.5 Price by Engine Size

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='engineSize', y='price')
plt.title("Price Distribution by Engine Size")
plt.tight_layout()
plt.show()

### 2.6 Price by Transmission Type

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='transmission', y='price')
plt.title("Price Distribution by Transmission")
plt.tight_layout()
plt.show()

### 2.7 Price by Fuel Type

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='fuelType', y='price')
plt.title("Price Distribution by Fuel Type")
plt.tight_layout()
plt.show()

### 2.8 Price by Model

In [ ]:
plt.figure(figsize=(16, 5))
sns.boxplot(data=df, x='model', y='price')
plt.xticks(rotation=90)
plt.title("Price Distribution by Model")
plt.tight_layout()
plt.show()

## 3. Feature & Target Split

In [ ]:
X = df.drop(columns=['price'], axis=1)
y = df['price']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

## 4. Model 1 — One-Hot Encoding + Linear Regression

One-Hot Encoding creates a separate binary column for each category. It avoids any implied ordinal relationship between categories.

### 4.1 One-Hot Encode Categorical Features

In [ ]:
X_ohe = pd.get_dummies(X, columns=['model', 'transmission', 'fuelType'], drop_first=True)
X_ohe = X_ohe.astype(int)

print(f"Shape after One-Hot Encoding: {X_ohe.shape}")
print(f"\nSample columns: {list(X_ohe.columns[:10])} ...")

### 4.2 Scale Numerical Features

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize']

scaler_ohe = StandardScaler()
X_ohe_scaled = X_ohe.copy()
X_ohe_scaled[numerical_cols] = scaler_ohe.fit_transform(X_ohe_scaled[numerical_cols])

print("Scaling complete.")
X_ohe_scaled.head(3)

### 4.3 Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train_ohe, X_test_ohe, y_train_ohe, y_test_ohe = train_test_split(
    X_ohe_scaled, y, test_size=0.20, random_state=42
)

print(f"Training set size : {X_train_ohe.shape[0]}")
print(f"Test set size     : {X_test_ohe.shape[0]}")

### 4.4 Train Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

model_ohe = LinearRegression()
model_ohe.fit(X_train_ohe, y_train_ohe)
print("Model 1 (One-Hot Encoding) trained.")

### 4.5 Evaluate Model 1

In [ ]:
y_pred_ohe = model_ohe.predict(X_test_ohe)

r2_ohe = r2_score(y_test_ohe, y_pred_ohe)
mae_ohe = mean_absolute_error(y_test_ohe, y_pred_ohe)
rmse_ohe = np.sqrt(mean_squared_error(y_test_ohe, y_pred_ohe))

n_ohe = X_test_ohe.shape[0]
p_ohe = X_test_ohe.shape[1]
adjusted_r2_ohe = 1 - ((1 - r2_ohe) * (n_ohe - 1) / (n_ohe - p_ohe - 1))

print("=== Model 1: One-Hot Encoding ===")
print(f"R²            : {r2_ohe:.4f}")
print(f"Adjusted R²   : {adjusted_r2_ohe:.4f}")
print(f"MAE           : {mae_ohe:.2f}")
print(f"RMSE          : {rmse_ohe:.2f}")

### 4.6 Actual vs Predicted Plot — Model 1

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_ohe, y_pred_ohe, alpha=0.3, color='steelblue')
plt.plot([y_test_ohe.min(), y_test_ohe.max()],
         [y_test_ohe.min(), y_test_ohe.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Model 1 (OHE): Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Model 2 — Label Encoding + Linear Regression

Label Encoding assigns an integer to each category. It's simpler and produces fewer columns, but may introduce unintended ordinal relationships.

### 5.1 Label Encode Categorical Features

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_le = X.copy()
# Normalise text first to avoid case/whitespace issues
for col in ['model', 'transmission', 'fuelType']:
    X_le[col] = X_le[col].astype(str).str.strip().str.lower()

label_encoders = {}
for col in ['model', 'transmission', 'fuelType']:
    le = LabelEncoder()
    X_le[col] = le.fit_transform(X_le[col])
    label_encoders[col] = le

print(f"Shape after Label Encoding: {X_le.shape}")
X_le.head(3)

### 5.2 Scale All Features

In [ ]:
scaler_le = StandardScaler()
X_le_scaled = X_le.copy()
X_le_scaled[X_le_scaled.columns] = scaler_le.fit_transform(X_le_scaled)

print("Scaling complete.")
X_le_scaled.head(3)

### 5.3 Train / Test Split

In [ ]:
X_train_le, X_test_le, y_train_le, y_test_le = train_test_split(
    X_le_scaled, y, test_size=0.20, random_state=42
)

print(f"Training set size : {X_train_le.shape[0]}")
print(f"Test set size     : {X_test_le.shape[0]}")

### 5.4 Train Linear Regression

In [ ]:
model_le = LinearRegression()
model_le.fit(X_train_le, y_train_le)
print("Model 2 (Label Encoding) trained.")

### 5.5 Evaluate Model 2

In [ ]:
y_pred_le = model_le.predict(X_test_le)

r2_le = r2_score(y_test_le, y_pred_le)
mae_le = mean_absolute_error(y_test_le, y_pred_le)
rmse_le = np.sqrt(mean_squared_error(y_test_le, y_pred_le))

n_le = X_test_le.shape[0]
p_le = X_test_le.shape[1]
adjusted_r2_le = 1 - ((1 - r2_le) * (n_le - 1) / (n_le - p_le - 1))

print("=== Model 2: Label Encoding ===")
print(f"R²            : {r2_le:.4f}")
print(f"Adjusted R²   : {adjusted_r2_le:.4f}")
print(f"MAE           : {mae_le:.2f}")
print(f"RMSE          : {rmse_le:.2f}")

### 5.6 Actual vs Predicted Plot — Model 2

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test_le, y_pred_le, alpha=0.3, color='darkorange')
plt.plot([y_test_le.min(), y_test_le.max()],
         [y_test_le.min(), y_test_le.max()],
         'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Model 2 (Label Enc): Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Encoding Strategy': ['One-Hot Encoding', 'Label Encoding'],
    'R²':              [round(r2_ohe, 4),          round(r2_le, 4)],
    'Adjusted R²':     [round(adjusted_r2_ohe, 4), round(adjusted_r2_le, 4)],
    'MAE':             [round(mae_ohe, 2),          round(mae_le, 2)],
    'RMSE':            [round(rmse_ohe, 2),         round(rmse_le, 2)],
    'Num Features':    [X_ohe_scaled.shape[1],      X_le_scaled.shape[1]]
})

print(comparison.to_string(index=False))

In [ ]:
# Visual comparison of R² scores
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics = ['R²', 'Adjusted R²']
for i, metric in enumerate(metrics):
    vals = comparison[metric].values
    bars = axes[i].bar(
        comparison['Encoding Strategy'], vals,
        color=['steelblue', 'darkorange'], edgecolor='black', width=0.5
    )
    axes[i].set_title(f'{metric} Comparison')
    axes[i].set_ylabel(metric)
    axes[i].set_ylim(0, 1)
    for bar, val in zip(bars, vals):
        axes[i].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontweight='bold'
        )

plt.suptitle("Model Comparison: One-Hot vs Label Encoding", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Conclusion

| Encoding | R² | Adjusted R² | Num Features |
|---|---|---|---|
| One-Hot Encoding | higher (typically) | accounts for extra features | ~34 |
| Label Encoding | comparable | same feature count penalty | 8 |

**Key Takeaways:**
- **One-Hot Encoding** is generally preferred for linear models since it doesn't impose any ordinal relationship on nominal categories like car model names.
- **Label Encoding** is simpler (fewer columns) and can work well with tree-based models, but for linear regression it may reduce accuracy because it implies numeric ordering between categories.
- The R² score tells us how much variance in car price the model explains. An R² of ~0.85+ would be a strong result for this dataset.